# GEMS | GeoRegime V2 — Discovery Recovery & Error Atlas

**Existing checkpoints only · no retraining · new cached full-area inference · rigorous holdout audit · scientifically relevant figures · guarded optional submission.**

This is a fixed notebook skeleton. Claude Code must implement the functions under `gems_bench/georegime_recovery/`; do not change the section order, documentation, or public function names without documenting a strictly necessary compatibility issue. All figures must be saved as PNG and SVG and tables as CSV and Parquet where appropriate. No previous model predictions or user results should be altered in place.

**Known diagnostic context:** GeoRegime V1 leaderboard 0.03; geographic proxy DTI 0.09958; narrow discovery proxy 0.10064; 72.1% of non-known-label prediction mass lies within 1 km of mapped faults. Those observations are not a replacement for artifact-level verification.


## 00 — Configuration and immutable inputs

Declare the existing run, Atlas root, isolated output namespace, GPU-inference toggle, modest postprocessing candidates, and output policy. Never use hidden labels or overwrite V1.


In [ ]:
from pathlib import Path
CFG = {
    "drive_root": "/content/drive/MyDrive/GEMS_2",
    "atlas_root": "GEMS_GEOLOGICAL_DATA_ATLAS_V1",
    "source_run": "GEMS_GEOREGIME_V1/runs/8dfa3b352417",
    "output_root": "GEMS_GEOREGIME_RECOVERY_V2",
    "scratch_root": "/content/georegime_recovery_v2",
    "repo": "mybenkhadda/GEMS---drivendata",
    "ref": "main",
    "seed": 2026,
    "train_models": False,
    "reuse_valid_cache": True,
    "compute_full_area_holdout_inference": True,
    "save_png": True,
    "save_svg": True,
    "save_csv": True,
    "save_parquet": True,
    "enable_submission_export": False,
}
assert CFG["train_models"] is False


## 01 — Mount Drive, pin code, initialize the recovery run

Source files remain unchanged. The implementation must not silently switch to another trained run. All derived outputs use a new namespace.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Claude: implement a self-contained, pinned-source bootstrap with explicit errors.
from gems_bench.georegime_recovery import RecoveryRun
RUN = RecoveryRun(CFG)
print(RUN.describe())


## 02 — Preflight: every artifact and exact provenance

Validate all 7 checkpoints, normalization JSONs, V1 prediction files, Atlas tensors and template; inventory missing/optional/derivable artifacts. No synthetic fallbacks.


In [ ]:
PREFLIGHT = RUN.preflight()
display(PREFLIGHT.table)
RUN.save_stage_manifest("preflight", PREFLIGHT)


## 03 — Baseline: prediction confidence and spatial budget

Quantiles, confidence concentration, total probability mass, area fractions above thresholds, reliability of nodata and explicit table exports.


In [ ]:
BASELINE = RUN.baseline_diagnostics()
display(BASELINE.summary)
RUN.plot_group("prediction_distribution")


## 04 — Mapped-fault proximity without using distances as predictions

Spatial mass distribution at 0–300m, 300–1000m, 1–3km, >3km; enrichment ratios and maps. Label proximity is diagnostic only.


In [ ]:
PROXIMITY = RUN.known_fault_proximity_diagnostics()
display(PROXIMITY.table)
RUN.plot_group("known_fault_proximity")


## 05 — Audit actual training-mask provenance

Reconstruct V1 geo0..2 and disc0..2 masks from checked-out source and original configuration. Verify each checkpoint train_mask_sha; fail on mismatch. Detail overlap, contamination, and holdout fragmentation.


In [ ]:
SPLITS = RUN.audit_original_splits()
display(SPLITS.unit_table)
RUN.plot_group("split_integrity")


## 06 — Re-score original geographic and narrow discovery predictions

Reproduce existing V1 proxy DTI components. Save exact scorer settings, exclusion conventions and discrepancy report before any new scoring.


In [ ]:
ORIGINAL = RUN.recompute_original_metrics()
display(ORIGINAL.components)
RUN.plot_group("original_dti")


## 07 — Full-area inference for existing CV/discovery checkpoints

No retraining: run each saved checkpoint over its supported full eligible area. New filename and ROI-specific fingerprint; do not call the V1 ROI-unsafe cache path. Atomic per-window cache/resume.


In [ ]:
FULL = RUN.ensure_full_area_predictions()
display(FULL.artifact_table)
RUN.plot_group("full_area_coverage")


## 08 — Full-area pseudo-discovery validation

Held truth is original withheld mapped systems only; other unknown faults remain unlabeled. Score exact training-visible known-fault exclusion, never use held labels for transform selection on their own fold.


In [ ]:
DISC = RUN.score_full_area_holdouts()
display(DISC.components)
RUN.plot_group("full_area_dti")


## 09 — Pixel-level TP/FP/FN support maps

Compute correct 300m weighted support, false-positive mass, missed-truth support using the repository metric kernel and exact exclusions; aggregate by zones and fault system.


In [ ]:
ERRORS = RUN.spatial_error_atlas()
display(ERRORS.region_table)
RUN.plot_group("error_atlas")


## 10 — Global head vs each specialist vs final mixture

Re-infer the saved checkpoints or cache each head separately: global, expert0, expert1, router weights, mixture strength, mixed score. Quantify expert diversity and collapse.


In [ ]:
EXPERTS = RUN.expert_diagnostics()
display(EXPERTS.summary)
RUN.plot_group("expert_complementarity")


## 11 — Cross-modal structure and hidden fault risk

Overlay label-free Atlas magnetic/gravity/terrain evidence, long-range gravity context, lineaments. Separate visible mapped-fault proximity from input-only geology.


In [ ]:
GEOLOGY = RUN.geological_disagreement_diagnostics()
display(GEOLOGY.summary)
RUN.plot_group("geological_disagreement")


## 12 — Fault length, orientation and spatial stratification

Measure short/medium/long fault recall, mapped fault strike and tortuosity, edge orientation agreement, and region-wise performance with coverage-aware denominators.


In [ ]:
STRATA = RUN.structural_stratification()
display(STRATA.table)
RUN.plot_group("structural_strata")


## 13 — Lineament proposal audit — no pseudo-labeling

Audit 184 Atlas lineament proposals, prioritize by label-free continuity/evidence with uncertainty, and overlay model scores; no automatic positive training labels.


In [ ]:
CANDIDATES = RUN.candidate_lineament_review()
display(CANDIDATES.top_candidates)
RUN.plot_group("candidate_lineaments")


## 14 — Small fixed postprocessing shortlist — NOT a leaderboard sweep

Compare identity and at most 3 predeclared physically defensible continuous-score transforms; keep known-fault labels OUT of inference features. No arbitrary 1km masking, no top-1% binary default.


In [ ]:
TRANSFORMS = RUN.evaluate_fixed_transform_shortlist()
display(TRANSFORMS.components_by_fold)
RUN.plot_group("transform_tradeoffs")


## 15 — Cross-fitted decisions and uncertainty

Choose any transform using other holdout folds; report untouched held-out components, fold variance, regional effects, and whether the decision is stable. Fall back to identity unless justified.


In [ ]:
DECISION = RUN.crossfit_recovery_decision()
display(DECISION.decision_table)
RUN.plot_group("crossfit_uncertainty")


## 16 — Optional guarded export: new continuous GeoTIFF

Only user opt-in AND full audit passed AND cross-fitted improvement stable. Export a NEW file and validate [0,1] finite stored values, template CRS/transform/mask, SHA256. No auto-upload.


In [ ]:
if CFG["enable_submission_export"]:
    SUBMISSION = RUN.export_guarded_submission(DECISION)
    display(SUBMISSION.validation_table)
    print(SUBMISSION.path)
else:
    print("Export disabled. Review cross-fitted diagnostics before enabling.")


## 17 — Final scientific report and artifact handoff

Summary of what is observed vs inferred; metrics in separate geographic/narrow discovery/full-area discovery contexts; all figure/table paths; blockers and next action.


In [ ]:
REPORT = RUN.build_final_report()
display(REPORT.summary)
display(RUN.artifact_inventory())
print("Recovery artifacts:", RUN.output_root)
